# Qwen Free-Form Instruction Parsing

## Purpose

This notebook tests whether Qwen2-VL-2B can understand a free-form instruction and identify which region should use which artist's style.

The correct answer is not shown to the model. Qwen only receives the image and the instruction.

## What it does

1. Uses all 200 samples from `instructions_200.json`.
2. Creates four different phrasings for each instruction.
3. Asks Qwen to return the region-artist pairs as JSON.
4. Compares the predictions with the correct answers.
5. Reports region accuracy, artist accuracy, exact-match accuracy, and swapped-style errors.
6. Calculates Wilson 95% confidence intervals.


In [ ]:
import os, sys, re, json, glob, time, random, hashlib, unicodedata, gc
from datetime import datetime, timezone

ON_COLAB = 'google.colab' in sys.modules
if ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    from IPython.display import display, Javascript
    display(Javascript('function KA(){document.querySelector("colab-toolbar-button#connect")?.click();} setInterval(KA,60000);'))

CONFIG = {
    'dataset_root': '/content/drive/MyDrive',
    'categories': ('City', 'Landscape', 'Nature', 'Sunset', 'Winter'),
    'design_table_path': '/content/drive/MyDrive/instructions_200.json',
    'allowed_styles': ['Van Gogh', 'Cezanne', 'Monet', 'Kandinsky', 'Delaunay'],
    'style_name_aliases': {'Picasso': 'Delaunay'},   # applies to the design table only
    'qwen_model': 'Qwen/Qwen2-VL-2B-Instruct',
    'qwen_max_tokens': 200,
    'qwen_max_pixels': 512 * 512,
    'samples_per_category': 40,
    'sample_ids_override': None,
    'sample_seed': 42,
    'phrasings': ['direct', 'reversed', 'casual_fullname', 'style_first'],
    'give_artist_list': True,
    'work_dir': '/content/drive/MyDrive/qwen_free_form_test_200',
}
RUN_ID = datetime.now(timezone.utc).strftime('freeform_%Y%m%dT%H%M%SZ')
print('RUN_ID:', RUN_ID)

Mounted at /content/drive


<IPython.core.display.Javascript object>

RUN_ID: freeform_20260920T174815Z


In [ ]:

import subprocess
if ON_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'transformers>=4.45', 'accelerate', 'qwen-vl-utils'], check=True)
import torch
HAS_CUDA = torch.cuda.is_available()
print('CUDA:', HAS_CUDA)

CUDA: True


## 1. Design table and subset (same design data as the existing notebook)

In [ ]:
def parse_instruction(instruction):
    # copied from the existing notebook so the design table is read identically
    pairs = []
    for region, style in re.findall(
            r'(?:the\s+)?([a-zA-Z ]+?)\s+like\s+([A-Za-z_ ]+?)(?:\s+and\s+|\s*,\s*|$)',
            instruction, re.IGNORECASE):
        region = re.sub(r'^(style|make|render|paint)\s+', '', region.strip(), flags=re.I).strip()
        region = re.sub(r'^the\s+', '', region, flags=re.I).strip()
        style = style.strip().replace('_', ' ')
        if region and style:
            pairs.append({'region': region, 'style': style})
    return pairs


def load_design_table(path):
    with open(path) as f:
        samples = json.load(f)
    rows = []
    for s in samples:
        by_region = {p['region']: p['style'] for p in parse_instruction(s['instruction'])}
        r1, r2 = s['regions'][0], s['regions'][1]
        al = CONFIG['style_name_aliases']
        s1 = al.get(by_region.get(r1), by_region.get(r1))
        s2 = al.get(by_region.get(r2), by_region.get(r2))
        rows.append({'sample_id': s['id'], 'region1': r1, 'region2': r2, 'style1': s1, 'style2': s2})
    return rows


def validate_design_table(rows):
    problems = []
    ids = [r['sample_id'] for r in rows]
    if len(set(ids)) != len(ids):
        problems.append('duplicate sample ids')
    for r in rows:
        for k in ('style1', 'style2'):
            if r[k] not in CONFIG['allowed_styles']:
                problems.append(f"{r['sample_id']}: {k}={r[k]!r} not in allowed styles")
        if r['region1'] == r['region2']:
            problems.append(f"{r['sample_id']}: identical regions")
    return problems


PREFIX_TO_CATEGORY = {c[0].upper(): c for c in CONFIG['categories']}


def select_subset(design_by_id, per_cat, seed, override=None):
    if override:
        return sorted(override)
    by_cat = {}
    for sid in sorted(design_by_id):
        by_cat.setdefault(PREFIX_TO_CATEGORY[sid[0].upper()], []).append(sid)
    rng = random.Random(seed)
    chosen = []
    for cat in CONFIG['categories']:
        ids = by_cat.get(cat, [])
        chosen += sorted(rng.sample(ids, min(per_cat, len(ids))))
    return chosen

In [ ]:
design_rows = load_design_table(CONFIG['design_table_path'])
problems = validate_design_table(design_rows)
assert not problems, problems
DESIGN_BY_ID = {r['sample_id']: r for r in design_rows}
SAMPLE_IDS = select_subset(DESIGN_BY_ID, CONFIG['samples_per_category'],
                           CONFIG['sample_seed'], CONFIG['sample_ids_override'])
print(f'{len(SAMPLE_IDS)} samples selected from {len(design_rows)}:')
print(', '.join(SAMPLE_IDS))

200 samples selected from 200:
C001, C002, C003, C004, C005, C006, C007, C008, C009, C010, C011, C012, C013, C014, C015, C016, C017, C018, C019, C020, C021, C022, C023, C024, C025, C026, C027, C028, C029, C030, C031, C032, C033, C034, C035, C036, C037, C038, C039, C040, L001, L002, L003, L004, L005, L006, L007, L008, L009, L010, L011, L012, L013, L014, L015, L016, L017, L018, L019, L020, L021, L022, L023, L024, L025, L026, L027, L028, L029, L030, L031, L032, L033, L034, L035, L036, L037, L038, L039, L040, N001, N002, N003, N004, N005, N006, N007, N008, N009, N010, N011, N012, N013, N014, N015, N016, N017, N018, N019, N020, N021, N022, N023, N024, N025, N026, N027, N028, N029, N030, N031, N032, N033, N034, N035, N036, N037, N038, N039, N040, S001, S002, S003, S004, S005, S006, S007, S008, S009, S010, S011, S012, S013, S014, S015, S016, S017, S018, S019, S020, S021, S022, S023, S024, S025, S026, S027, S028, S029, S030, S031, S032, S033, S034, S035, S036, S037, S038, S039, S040, W001, W00

## 2. Image discovery

In [ ]:
def discover_category_folders(root, categories):
    entries = [e for e in os.listdir(root) if os.path.isdir(os.path.join(root, e))]
    norm = {c.strip().lower(): c for c in categories}
    folders = {}
    for e in entries:
        k = e.strip().lower()
        if k in norm:
            folders[norm[k][0].upper()] = os.path.join(root, e)
    missing = [c for c in categories if c[0].upper() not in folders]
    assert not missing, f'category folders not found: {missing}; present: {sorted(entries)[:20]}'
    return folders


def find_content_image(sample_id, folders):
    folder = folders[sample_id[0].upper()]
    files = os.listdir(folder)
    m = sorted(f for f in files if re.match(rf'^{re.escape(sample_id)}[._]', f, re.IGNORECASE))
    if m:
        return os.path.join(folder, m[0])
    m = sorted(f for f in files if sample_id.lower() in f.lower())
    if m:
        return os.path.join(folder, m[0])
    raise FileNotFoundError(f'no image for {sample_id} in {folder}; first files: {sorted(files)[:8]}')

In [ ]:
CATEGORY_FOLDERS = discover_category_folders(CONFIG['dataset_root'], CONFIG['categories'])
IMAGE_PATH = {sid: find_content_image(sid, CATEGORY_FOLDERS) for sid in SAMPLE_IDS}   # pre-flight
print('all', len(IMAGE_PATH), 'images found')

all 200 images found


## 3. Free-form phrasings
Four phrasings per sample. They differ in word order, verb, and whether the artist is written in full, so the model must
actually read the request rather than copy a fixed pattern. The **reversed** and **style-first** phrasings mention the second region first,
which tests whether each artist is bound to the right region.

In [ ]:
DISPLAY = {   # [short name, fuller name]
    'Van Gogh':  ['Van Gogh', 'Vincent van Gogh'],
    'Cezanne':['Cezanne', 'Paul C\u00e9zanne'],
    'Monet':['Monet', 'Claude Monet'],
    'Kandinsky': ['Kandinsky', 'Wassily Kandinsky'],
    'Delaunay':['Delaunay', 'Delaunay'],
}


def make_phrasings(row):
    r1, r2, a1, a2 = row['region1'], row['region2'], row['style1'], row['style2']
    s = lambda a: DISPLAY[a][0]
    f = lambda a: DISPLAY[a][1]
    return {
        'direct':f"Make the {r1} look like a {s(a1)} painting and the {r2} look like a {s(a2)} painting.",
        'reversed':f"Paint the {r2} in the style of {s(a2)}, and give the {r1} the style of {s(a1)}.",
        'casual_fullname': f"I'd like the {r1} to feel like the work of {f(a1)}, while the {r2} should look like something {f(a2)} would have painted.",
        'style_first':f"{s(a2)} for the {r2}, {s(a1)} for the {r1}. Leave everything else as it is.",
    }


# show what the four phrasings look like for the first selected sample
_demo = DESIGN_BY_ID[SAMPLE_IDS[0]]
print('design (hidden from the model):', _demo)
for k, v in make_phrasings(_demo).items():
    print(f'  {k:16s}: {v}')

design (hidden from the model): {'sample_id': 'C001', 'region1': 'sky', 'region2': 'buildings', 'style1': 'Cezanne', 'style2': 'Van Gogh'}
  direct          : Make the sky look like a Cezanne painting and the buildings look like a Van Gogh painting.
  reversed        : Paint the buildings in the style of Van Gogh, and give the sky the style of Cezanne.
  casual_fullname : I'd like the sky to feel like the work of Paul Cézanne, while the buildings should look like something Vincent van Gogh would have painted.
  style_first     : Van Gogh for the buildings, Cezanne for the sky. Leave everything else as it is.


## 4. Prompt, JSON parsing and scoring
The prompt contains the free-form request and (optionally) the list of allowed artist names. It contains **no region names from the design table**
other than those that appear inside the request itself.

In [ ]:
SYSTEM_PROMPT = (
    "You extract structured information from image-editing requests. You are given an image and a request "
    "that assigns the painting style of an artist to each of two regions of the image. Read the request and "
    "report which region gets which artist. Also say whether each region is visible in the image. "
    "Answer with a JSON object only."
)


def build_user_prompt(request):
    artist_line = ''
    if CONFIG['give_artist_list']:
        artist_line = 'Allowed artists: ' + ', '.join(CONFIG['allowed_styles']) + '.\n'
    return (
        f'Request: "{request}"\n'
        f'{artist_line}'
        'Return ONLY a JSON object of this form (no markdown, no other text):\n'
        '{"assignments": [{"region": "<region noun as named in the request>", "artist": "<artist name>", '
        '"visible_in_image": <true or false>}, {"region": "...", "artist": "...", "visible_in_image": ...}]}'
    )


def extract_json(raw):
    cleaned = re.sub(r'^```(?:json)?\s*|\s*```$', '', raw.strip(), flags=re.MULTILINE).strip()
    try:
        return json.loads(cleaned), None
    except json.JSONDecodeError:
        m = re.search(r'\{.*\}', cleaned, re.DOTALL)
        if not m:
            return None, 'no JSON object found'
        try:
            return json.loads(m.group(0)), None
        except json.JSONDecodeError as e:
            return None, f'invalid JSON: {e}'


def get_assignments(obj):
    if not isinstance(obj, dict) or not isinstance(obj.get('assignments'), list):
        return None
    out = []
    for a in obj['assignments']:
        if isinstance(a, dict):
            out.append({'region': str(a.get('region', '')), 'artist': str(a.get('artist', '')),
                        'visible': a.get('visible_in_image')})
    return out

In [ ]:
def strip_accents(s):
    return ''.join(c for c in unicodedata.normalize('NFKD', s) if not unicodedata.combining(c))


def norm_text(s):
    s = strip_accents(str(s)).lower().strip()
    s = re.sub(r'[^a-z ]', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()


def norm_region(s):
    s = norm_text(s)
    s = re.sub(r'^(the|a|an)\s+', '', s)
    return s


def singular(s):
    return s[:-1] if s.endswith('s') and not s.endswith('ss') else s


def region_strict(pred, gold):
    return singular(norm_region(pred)) == singular(norm_region(gold))


def region_lenient(pred, gold):
    p, g = singular(norm_region(pred)), singular(norm_region(gold))
    return bool(p) and bool(g) and (p == g or p in g or g in p)


ARTIST_ALIASES = {
    'van gogh': 'Van Gogh', 'vincent van gogh': 'Van Gogh', 'vangogh': 'Van Gogh', 'gogh': 'Van Gogh',
    'cezanne': 'Cezanne', 'paul cezanne': 'Cezanne',
    'monet': 'Monet', 'claude monet': 'Monet',
    'kandinsky': 'Kandinsky', 'wassily kandinsky': 'Kandinsky',
    'delaunay': 'Delaunay', 'robert delaunay': 'Delaunay', 'sonia delaunay': 'Delaunay',
}   # deliberately no 'picasso': a model answering Picasso is a wrong answer


def canon_artist(s):
    k = norm_text(s)
    if k in ARTIST_ALIASES:
        return ARTIST_ALIASES[k]
    for alias, name in ARTIST_ALIASES.items():   # e.g. "in the style of monet"
        if alias in k:
            return name
    return None


def score_assignments(row, assignments):
    gold = [(row['region1'], row['style1']), (row['region2'], row['style2'])]
    res = {'n_assignments': len(assignments), 'region_ok': [], 'region_strict_ok': [], 'style_ok': [],
           'pred_styles': []}
    used = set()
    for gr, gs in gold:
        idx = next((i for i, p in enumerate(assignments) if i not in used and region_lenient(p['region'], gr)), None)
        if idx is None:
            res['region_ok'].append(False); res['region_strict_ok'].append(False)
            res['style_ok'].append(False); res['pred_styles'].append(None)
            continue
        used.add(idx)
        p = assignments[idx]
        ps = canon_artist(p['artist'])
        res['region_ok'].append(True)
        res['region_strict_ok'].append(region_strict(p['region'], gr))
        res['style_ok'].append(ps == gs)
        res['pred_styles'].append(ps)
    res['exact'] = (len(assignments) == 2 and all(res['region_ok']) and all(res['style_ok']))
    res['strict_exact'] = res['exact'] and all(res['region_strict_ok'])
    both_found = all(res['region_ok'])
    res['swapped'] = bool(both_found and res['pred_styles'][0] == gold[1][1]
                          and res['pred_styles'][1] == gold[0][1] and gold[0][1] != gold[1][1])
    res['visible_both'] = bool(len(assignments) == 2 and all(a['visible'] is True for a in assignments))
    return res

In [ ]:
_row = {'sample_id': 'T001', 'region1': 'sky', 'region2': 'buildings', 'style1': 'Van Gogh', 'style2': 'Cezanne'}

def _mk(*pairs):
    return {'assignments': [{'region': r, 'artist': a, 'visible_in_image': True} for r, a in pairs]}

# perfect
s = score_assignments(_row, get_assignments(_mk(('sky', 'Van Gogh'), ('buildings', 'Cezanne'))))
assert s['exact'] and s['strict_exact'] and not s['swapped']
# order reversed in the output is still correct
s = score_assignments(_row, get_assignments(_mk(('buildings', 'Cezanne'), ('sky', 'Van Gogh'))))
assert s['exact']
# plural / article / accent / full-name normalisation
s = score_assignments(_row, get_assignments(_mk(('the Sky', 'Vincent van Gogh'), ('building', 'Paul C\u00e9zanne'))))
assert s['exact'], s
# swapped styles are detected and are wrong
s = score_assignments(_row, get_assignments(_mk(('sky', 'Cezanne'), ('buildings', 'Van Gogh'))))
assert (not s['exact']) and s['swapped'] and all(s['region_ok']) and not any(s['style_ok'])
# wrong artist (Picasso is not accepted)
s = score_assignments(_row, get_assignments(_mk(('sky', 'Picasso'), ('buildings', 'Cezanne'))))
assert (not s['exact']) and s['style_ok'] == [False, True]
# missing region
s = score_assignments(_row, get_assignments(_mk(('sky', 'Van Gogh'), ('trees', 'Cezanne'))))
assert (not s['exact']) and s['region_ok'] == [True, False]
# extra assignment
s = score_assignments(_row, get_assignments(_mk(('sky', 'Van Gogh'), ('buildings', 'Cezanne'), ('street', 'Monet'))))
assert not s['exact'] and s['n_assignments'] == 3
# duplicate item cannot satisfy both regions
s = score_assignments(_row, get_assignments(_mk(('sky', 'Van Gogh'), ('sky', 'Van Gogh'))))
assert not s['exact']
# JSON extraction: fenced, prefixed, and broken
obj, err = extract_json('```json\n{"assignments": []}\n```'); assert obj == {'assignments': []} and err is None
obj, err = extract_json('Sure! {"assignments": []} hope this helps'); assert obj == {'assignments': []}
obj, err = extract_json('no json here'); assert obj is None
obj, err = extract_json('{"assignments": [}'); assert obj is None
assert get_assignments({'x': 1}) is None
# phrasings never contain the words 'like <style>' pattern of the templated instruction only, and all four differ
ph = make_phrasings(_row)
assert len(set(ph.values())) == 4
print('self-test passed: parsing, normalisation and scoring behave as intended')

self-test passed: parsing, normalisation and scoring behave as intended


## 5. Load Qwen2-VL-2B

In [ ]:
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
model = Qwen2VLForConditionalGeneration.from_pretrained(CONFIG['qwen_model'], torch_dtype='auto', device_map='auto')
processor = AutoProcessor.from_pretrained(CONFIG['qwen_model'], min_pixels=256 * 28 * 28,
                                          max_pixels=CONFIG['qwen_max_pixels'])
print('loaded', CONFIG['qwen_model'])
if HAS_CUDA:
    print(f'VRAM free: {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB')


def call_qwen(image_path, request):
    from qwen_vl_utils import process_vision_info
    user_prompt = build_user_prompt(request)
    messages = [
        {'role': 'system', 'content': [{'type': 'text', 'text': SYSTEM_PROMPT}]},
        {'role': 'user', 'content': [{'type': 'image', 'image': image_path},
                                     {'type': 'text', 'text': user_prompt}]},
    ]
    chat_text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[chat_text], images=image_inputs, videos=video_inputs,
                       padding=True, return_tensors='pt').to(model.device)
    t0 = time.time()
    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=CONFIG['qwen_max_tokens'], do_sample=False)  # greedy
    trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, out_ids)]
    text = processor.batch_decode(trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
    latency = time.time() - t0
    del inputs, out_ids, trimmed
    if HAS_CUDA:
        torch.cuda.empty_cache()
    gc.collect()
    return text, user_prompt, latency

config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/56.4k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

loaded Qwen/Qwen2-VL-2B-Instruct
VRAM free: 37.48 GB


## 6. Smoke test on one sample and one phrasing

In [ ]:
_sid = SAMPLE_IDS[0]
_row = DESIGN_BY_ID[_sid]
_req = make_phrasings(_row)['reversed']
raw, prompt, lat = call_qwen(IMAGE_PATH[_sid], _req)
print('request :', _req)
print('gold    :', [(_row['region1'], _row['style1']), (_row['region2'], _row['style2'])])
print('raw     :', raw)
obj, err = extract_json(raw)
asg = get_assignments(obj) if obj else None
print('parsed  :', asg, '| error:', err)
if asg is not None:
    print('score   :', score_assignments(_row, asg))
print(f'latency : {lat:.1f}s')

request : Paint the buildings in the style of Van Gogh, and give the sky the style of Cezanne.
gold    : [('sky', 'Cezanne'), ('buildings', 'Van Gogh')]
raw     : ```json
{
  "assignments": [
    {
      "region": "city buildings",
      "artist": "Van Gogh",
      "visible_in_image": true
    },
    {
      "region": "sky",
      "artist": "Cezanne",
      "visible_in_image": true
    }
  ]
}
```
parsed  : [{'region': 'city buildings', 'artist': 'Van Gogh', 'visible': True}, {'region': 'sky', 'artist': 'Cezanne', 'visible': True}] | error: None
score   : {'n_assignments': 2, 'region_ok': [True, True], 'region_strict_ok': [True, False], 'style_ok': [True, True], 'pred_styles': ['Cezanne', 'Van Gogh'], 'exact': True, 'strict_exact': False, 'swapped': False, 'visible_both': True}
latency : 5.4s


## 7. Main loop (checkpointed and resumable)

In [ ]:
OUT_DIR = CONFIG['work_dir']
os.makedirs(OUT_DIR, exist_ok=True)
CKPT = os.path.join(OUT_DIR, f"checkpoint_artistlist{int(CONFIG['give_artist_list'])}.json")


def load_ckpt():
    return json.load(open(CKPT)) if os.path.exists(CKPT) else []


def save_ckpt(records):
    tmp = CKPT + '.tmp'
    json.dump(records, open(tmp, 'w'))
    os.replace(tmp, CKPT)


records = load_ckpt()
done = {(r['sample_id'], r['phrasing']) for r in records}
todo = [(sid, ph) for sid in SAMPLE_IDS for ph in make_phrasings(DESIGN_BY_ID[sid]) if ph in CONFIG['phrasings']]
todo = [t for t in todo if t not in done]
print(f'{len(done)} calls already done, {len(todo)} to run (~{len(todo) * 5 / 60:.0f} min at ~5 s per call)')

t_start = time.time()
for n, (sid, ph) in enumerate(todo, 1):
    row = DESIGN_BY_ID[sid]
    request = make_phrasings(row)[ph]
    rec = {'sample_id': sid, 'category': PREFIX_TO_CATEGORY[sid[0].upper()], 'phrasing': ph,
           'request': request, 'artist_list_given': CONFIG['give_artist_list'],
           'gold_region1': row['region1'], 'gold_style1': row['style1'],
           'gold_region2': row['region2'], 'gold_style2': row['style2'],
           'run_id': RUN_ID, 'qwen_model': CONFIG['qwen_model']}
    try:
        raw, _, lat = call_qwen(IMAGE_PATH[sid], request)
        rec.update({'raw_output': raw, 'latency_s': lat})
        obj, err = extract_json(raw)
        asg = get_assignments(obj) if obj is not None else None
        if asg is None:
            rec.update({'parsed': False, 'parse_error': err or "missing 'assignments' list"})
        else:
            rec.update({'parsed': True, 'parse_error': None, 'assignments': asg})
            rec.update(score_assignments(row, asg))
    except Exception as e:
        rec.update({'raw_output': None, 'parsed': False, 'parse_error': f'{type(e).__name__}: {e}'})
    records.append(rec)
    if n % 10 == 0 or n == len(todo):
        save_ckpt(records)
        ok = sum(1 for r in records if r.get('exact'))
        print(f'{n}/{len(todo)} | exact so far {ok}/{len(records)} | {time.time() - t_start:.0f}s | checkpoint saved')
save_ckpt(records)
print('done:', len(records), 'calls')

800 calls already done, 0 to run (~0 min at ~5 s per call)
done: 800 calls


## 8. Results

In [ ]:
import pandas as pd

def wilson(k, n, z=1.96):
    if n == 0:
        return (float('nan'), float('nan'))
    p = k / n
    d = 1 + z * z / n
    c = p + z * z / (2 * n)
    h = z * ((p * (1 - p) / n + z * z / (4 * n * n)) ** 0.5)
    return ((c - h) / d, (c + h) / d)


df = pd.DataFrame(records)
for col in ('region_ok', 'style_ok', 'n_assignments', 'parsed', 'exact', 'strict_exact', 'swapped', 'visible_both'):
    if col not in df:
        df[col] = None
for col in ('exact', 'strict_exact', 'swapped', 'visible_both'):
    df[col] = df[col].fillna(False).astype(bool) if col in df else False
df['parsed'] = df['parsed'].fillna(False).astype(bool)
df['region_both'] = df['region_ok'].apply(lambda x: bool(isinstance(x, list) and all(x)))
df['style_both'] = df['style_ok'].apply(lambda x: bool(isinstance(x, list) and all(x)))
df['n_ok'] = df['n_assignments'].fillna(-1).astype(int).eq(2)

N = len(df)
def rate(mask):
    k = int(mask.sum()); lo, hi = wilson(k, len(mask))
    return f'{k}/{len(mask)} = {100 * k / max(len(mask), 1):.1f}%  (95% CI {100 * lo:.1f}-{100 * hi:.1f}%)'

print(f'RUN {RUN_ID} | calls: {N} | samples: {df.sample_id.nunique()} | artist list given: {CONFIG["give_artist_list"]}\n')
print('valid JSON with assignments :', rate(df.parsed))
print('exactly two assignments     :', rate(df.n_ok))
print('both regions recovered      :', rate(df.region_both))
print('both artists correct        :', rate(df.style_both))
print('FULL EXACT MATCH            :', rate(df.exact))
print('  ... with strict region names:', rate(df.strict_exact))
print('swapped-style errors        :', int(df.swapped.sum()), 'of', N)
print('both regions judged visible :', rate(df.visible_both))

RUN freeform_20260920T174815Z | calls: 800 | samples: 200 | artist list given: True

valid JSON with assignments : 800/800 = 100.0%  (95% CI 99.5-100.0%)
exactly two assignments     : 799/800 = 99.9%  (95% CI 99.3-100.0%)
both regions recovered      : 735/800 = 91.9%  (95% CI 89.8-93.6%)
both artists correct        : 693/800 = 86.6%  (95% CI 84.1-88.8%)
FULL EXACT MATCH            : 692/800 = 86.5%  (95% CI 84.0-88.7%)
  ... with strict region names: 679/800 = 84.9%  (95% CI 82.2-87.2%)
swapped-style errors        : 32 of 800
both regions judged visible : 798/800 = 99.8%  (95% CI 99.1-99.9%)


In [ ]:
by_phr = df.groupby('phrasing').agg(calls=('exact', 'size'), exact=('exact', 'mean'),
                                    region_both=('region_both', 'mean'), style_both=('style_both', 'mean'),
                                    swapped=('swapped', 'sum')).round(3)
by_cat = df.groupby('category').agg(calls=('exact', 'size'), exact=('exact', 'mean'),
                                    region_both=('region_both', 'mean'), style_both=('style_both', 'mean')).round(3)
print('BY PHRASING\n', by_phr.to_string(), '\n')
print('BY CATEGORY\n', by_cat.to_string(), '\n')

def error_type(r):
    if r.exact: return 'correct'
    if not r.parsed: return 'unparseable output'
    if not r.n_ok: return 'wrong number of assignments'
    if r.swapped: return 'styles swapped between regions'
    if not r.region_both: return 'region not recovered'
    return 'wrong artist'

df['error_type'] = df.apply(error_type, axis=1)
print('ERROR TYPES\n', df.error_type.value_counts().to_string(), '\n')

fails = df[~df.exact]
cols = ['sample_id', 'phrasing', 'request', 'error_type', 'raw_output']
print(f'FIRST {min(10, len(fails))} FAILURES')
for _, r in fails.head(10).iterrows():
    print(f"\n[{r.sample_id} / {r.phrasing}] {r.error_type}\n  request: {r.request}\n  gold   : ({r.gold_region1}, {r.gold_style1}), ({r.gold_region2}, {r.gold_style2})\n  raw    : {str(r.raw_output)[:300]}")

BY PHRASING
                  calls  exact  region_both  style_both  swapped
phrasing                                                       
casual_fullname    200   0.79        0.855       0.790       12
direct             200   0.85        0.905       0.850        8
reversed           200   0.94        0.955       0.940        0
style_first        200   0.88        0.960       0.885       12 

BY CATEGORY
            calls  exact  region_both  style_both
category                                        
City         160  0.881        0.912       0.881
Landscape    160  0.881        0.956       0.881
Nature       160  0.888        0.925       0.888
Sunset       160  0.938        0.981       0.938
Winter       160  0.738        0.819       0.744 

ERROR TYPES
 error_type
correct                           692
region not recovered               65
styles swapped between regions     32
wrong artist                       10
wrong number of assignments         1 

FIRST 10 FAILURES

[C001 / 

In [ ]:
tag = f"artistlist{int(CONFIG['give_artist_list'])}"
df.to_csv(os.path.join(OUT_DIR, f'results_{tag}.csv'), index=False)
json.dump({'run_id': RUN_ID, 'config': {k: v for k, v in CONFIG.items() if k != 'work_dir'},
           'n_calls': int(N), 'n_samples': int(df.sample_id.nunique()),
           'exact': int(df.exact.sum()), 'region_both': int(df.region_both.sum()),
           'style_both': int(df.style_both.sum()), 'swapped': int(df.swapped.sum()),
           'parsed': int(df.parsed.sum())},
          open(os.path.join(OUT_DIR, f'summary_{tag}.json'), 'w'), indent=2)
print('saved to', OUT_DIR)

saved to /content/drive/MyDrive/qwen_free_form_test_200


In [ ]:
import numpy as np, pandas as pd, re

rng = np.random.default_rng(42)
B = 5000

# sample-level (clustered) bootstrap: resample SAMPLES, keep their 4 phrasings together
def cluster_ci(frame, col):
    per = frame.groupby('sample_id')[col].agg(['sum', 'count'])
    s, c = per['sum'].to_numpy(float), per['count'].to_numpy(float)
    idx = rng.integers(0, len(s), size=(B, len(s)))
    est = s[idx].sum(1) / c[idx].sum(1)
    return s.sum() / c.sum(), np.percentile(est, 2.5), np.percentile(est, 97.5)

# three region-matching rules (the main run used 'lenient')
def region_word(pred, gold):   # gold word must appear as a whole word (sky != skyscrapers)
    return singular(norm_region(gold)) in {singular(w) for w in norm_region(pred).split()}

def exact_under(row, match):
    obj, _ = extract_json(row['raw_output']) if isinstance(row['raw_output'], str) else (None, None)
    asg = get_assignments(obj) if obj is not None else None
    if asg is None or len(asg) != 2:
        return False
    used = set()
    for gr, gs in ((row['gold_region1'], row['gold_style1']), (row['gold_region2'], row['gold_style2'])):
        i = next((k for k, p in enumerate(asg) if k not in used and match(p['region'], gr)), None)
        if i is None:
            return False
        used.add(i)
        if canon_artist(asg[i]['artist']) != gs:
            return False
    return True

df['ex_strict']  = df.apply(lambda r: exact_under(r, region_strict), axis=1)
df['ex_word']    = df.apply(lambda r: exact_under(r, region_word), axis=1)
df['ex_lenient'] = df.apply(lambda r: exact_under(r, region_lenient), axis=1)

print('EXACT MATCH, sample-level 95% CI, under three region rules')
for col, label in (('ex_strict', 'strict'), ('ex_word', 'word-level'), ('ex_lenient', 'lenient (main run)')):
    p, lo, hi = cluster_ci(df, col)
    print(f'  {label:20s} {100*p:5.1f}%   [{100*lo:.1f}, {100*hi:.1f}]')
print('  rows where lenient and word-level disagree:', int((df.ex_lenient != df.ex_word).sum()))

print('\nBY PHRASING (main rule, sample-level CI)')
for ph in sorted(df.phrasing.unique()):
    p, lo, hi = cluster_ci(df[df.phrasing == ph], 'ex_lenient')
    print(f'  {ph:16s} {100*p:5.1f}%   [{100*lo:.1f}, {100*hi:.1f}]')

print('\nSAMPLES BY NUMBER OF PHRASINGS (of 4) PARSED EXACTLY')
per = df.groupby('sample_id').agg(category=('category', 'first'), n_correct=('ex_lenient', 'sum'))
print(per.n_correct.value_counts().sort_index(ascending=False).to_string())
print('samples with 0 of 4:', ', '.join(per[per.n_correct == 0].index) or 'none')

# what did the model write instead of the region?
WHOLE = re.compile(r'\b(all|whole|entire|everything|image|picture|photo|scene)\b|[\u4e00-\u9fff]')
inv = []
for _, r in df[~df.ex_lenient].iterrows():
    obj, _ = extract_json(r['raw_output']) if isinstance(r['raw_output'], str) else (None, None)
    asg = get_assignments(obj) if obj is not None else None
    for p in (asg or []):
        gold = [r['gold_region1'], r['gold_region2']]
        if not any(region_lenient(p['region'], g) for g in gold):
            inv.append({'gold regions': ' / '.join(gold), 'predicted': p['region'],
                        'kind': 'whole-image answer' if WHOLE.search(str(p['region']).lower()) else 'other wording'})
inv = pd.DataFrame(inv, columns=['gold regions', 'predicted', 'kind'])
print('\nFAILURE INVENTORY: predicted regions matching neither gold region =', len(inv))
if len(inv):
    print(inv.kind.value_counts().to_string())
    print(inv.groupby(['kind', 'gold regions', 'predicted']).size().sort_values(ascending=False).head(30).to_string())

EXACT MATCH, sample-level 95% CI, under three region rules
  strict                84.9%   [81.9, 87.8]
  word-level            86.2%   [83.4, 89.0]
  lenient (main run)    86.5%   [83.6, 89.2]
  rows where lenient and word-level disagree: 2

BY PHRASING (main rule, sample-level CI)
  casual_fullname   79.0%   [73.0, 84.5]
  direct            85.0%   [80.0, 89.5]
  reversed          94.0%   [90.5, 97.0]
  style_first       88.0%   [83.5, 92.0]

SAMPLES BY NUMBER OF PHRASINGS (of 4) PARSED EXACTLY
n_correct
4    128
3     41
2     27
1      3
0      1
samples with 0 of 4: C003

FAILURE INVENTORY: predicted regions matching neither gold region = 84
kind
other wording         65
whole-image answer    19
kind                gold regions        predicted                     
other wording       sky / valley        landscape                         7
                    water / forest      river                             6
                    sky / forest        snowy landscape            

In [ ]:

import re, random

REGION_SYNONYMS = {
    'buildings': ['skyline', 'cityscape', 'skyscraper', 'city', 'architecture', 'downtown', 'tower'],
    'sky':       ['skies', 'clouds'],
    'mountains': ['mountain range', 'peaks', 'hills'],
    'water':     ['lake', 'river', 'sea', 'ocean', 'pond', 'stream'],
    'forest':    ['trees', 'woods', 'woodland'],
    'street':    ['road', 'pavement', 'sidewalk'],
    'rocks':     ['rock', 'boulders'],
}
ZH = {'天空': 'sky', '山脉': 'mountains', '森林': 'forest', '树林': 'forest', '湖': 'water', '河': 'water', '水': 'water',
      '建筑': 'buildings', '街道': 'street', '岩石': 'rocks', '山谷': 'valley', '草地': 'meadow'}
GENUINE_WHOLE = re.compile(r'\b(all|whole|entire|everything)\b|整个')

def region_post(pred, gold):
    p = str(pred)
    if GENUINE_WHOLE.search(p.lower()):
        return False
    for zh, en in ZH.items():
        if zh in p:
            p = p.replace(zh, ' ' + en + ' ')
    if region_word(p, gold):
        return True
    g = singular(norm_region(gold))
    pn = norm_region(p)
    return any(a in pn for a in REGION_SYNONYMS.get(g, REGION_SYNONYMS.get(g + 's', [])))

df['ex_post'] = df.apply(lambda r: exact_under(r, region_post), axis=1)
for col, label in (('ex_lenient', 'main rule'), ('ex_post', 'post hoc (synonyms + Chinese equivalents)')):
    p, lo, hi = cluster_ci(df, col)
    print(f'{label:45s} {100*p:5.1f}%   [{100*lo:.1f}, {100*hi:.1f}]')

# how many failures still have BOTH gold artists named somewhere (i.e. wording, not necessarily binding)?
def artist_set_ok(row):
    obj, _ = extract_json(row['raw_output']) if isinstance(row['raw_output'], str) else (None, None)
    asg = get_assignments(obj) if obj is not None else None
    if not asg:
        return False
    return {canon_artist(a['artist']) for a in asg} == {row['gold_style1'], row['gold_style2']}
df['artist_set_ok'] = df.apply(artist_set_ok, axis=1)
print('\nboth gold artists named (ignoring which region):', int(df.artist_set_ok.sum()), 'of', len(df))
print('failures (main rule) that still name both artists:', int((~df.ex_lenient & df.artist_set_ok).sum()))

# manual audit: read these yourself and count how many are wording-only vs a real wrong binding
random.seed(1)
pool = df[~df.ex_lenient & df.artist_set_ok].index.tolist()
sample = random.sample(pool, min(30, len(pool)))
print('\nMANUAL AUDIT SAMPLE (judge each: A = wording only, correct pairing; B = wrong pairing)\n')
for k, i in enumerate(sample, 1):
    r = df.loc[i]
    obj, _ = extract_json(r['raw_output'])
    pred = [(a['region'], a['artist']) for a in get_assignments(obj)]
    print(f"{k:2d}. [{r['sample_id']}/{r['phrasing']}]  request: {r['request']}")
    print(f"     gold: ({r['gold_region1']}, {r['gold_style1']}), ({r['gold_region2']}, {r['gold_style2']})   pred: {pred}")

main rule                                      86.5%   [83.5, 89.2]
post hoc (synonyms + Chinese equivalents)      87.9%   [85.1, 90.5]

both gold artists named (ignoring which region): 784 of 800
failures (main rule) that still name both artists: 92

MANUAL AUDIT SAMPLE (judge each: A = wording only, correct pairing; B = wrong pairing)

 1. [L001/style_first]  request: Monet for the mountains, Van Gogh for the sky. Leave everything else as it is.
     gold: (sky, Van Gogh), (mountains, Monet)   pred: [('mountains', 'Van Gogh'), ('sky', 'Monet')]
 2. [W013/direct]  request: Make the sky look like a Cezanne painting and the forest look like a Van Gogh painting.
     gold: (sky, Cezanne), (forest, Van Gogh)   pred: [('snowy forest', 'Cezanne'), ('cloudy sky', 'Van Gogh')]
 3. [C008/style_first]  request: Delaunay for the buildings, Kandinsky for the sky. Leave everything else as it is.
     gold: (sky, Kandinsky), (buildings, Delaunay)   pred: [('Philadelphia skyline', 'Delaunay'), ('Phi

In [ ]:

import os, json, time, gc, torch

assert CONFIG['give_artist_list'] is True, 'keep the same prompt setting as the image run'

SYSTEM_PROMPT_TEXT = (
    "You extract structured information from image-editing requests. The request assigns the painting style "
    "of an artist to each of two regions of an image (no image is provided; always set visible_in_image to true). "
    "Read the request and report which region gets which artist. Answer with a JSON object only."
)

def call_qwen_text(request):
    user_prompt = build_user_prompt(request)
    messages = [
        {'role': 'system', 'content': [{'type': 'text', 'text': SYSTEM_PROMPT_TEXT}]},
        {'role': 'user',   'content': [{'type': 'text', 'text': user_prompt}]},
    ]
    chat_text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[chat_text], padding=True, return_tensors='pt').to(model.device)
    t0 = time.time()
    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=CONFIG['qwen_max_tokens'], do_sample=False)
    trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, out_ids)]
    text = processor.batch_decode(trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
    lat = time.time() - t0
    del inputs, out_ids, trimmed
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    return text, lat

# smoke test on one request
_sid = SAMPLE_IDS[0]; _row = DESIGN_BY_ID[_sid]; _req = make_phrasings(_row)['reversed']
_raw, _lat = call_qwen_text(_req)
print('request:', _req)
print('gold   :', [(_row['region1'], _row['style1']), (_row['region2'], _row['style2'])])
print('raw    :', _raw)
print(f'latency: {_lat:.1f}s')

request: Paint the buildings in the style of Van Gogh, and give the sky the style of Cezanne.
gold   : [('sky', 'Cezanne'), ('buildings', 'Van Gogh')]
raw    : {
  "assignments": [
    {
      "region": "buildings",
      "artist": "Van Gogh",
      "visible_in_image": true
    },
    {
      "region": "sky",
      "artist": "Cezanne",
      "visible_in_image": true
    }
  ]
}
latency: 2.4s


In [ ]:

import os, json, time
import pandas as pd

OUT_TXT = '/content/drive/MyDrive/qwen_free_form_test_200_textonly'
os.makedirs(OUT_TXT, exist_ok=True)
CKPT_TXT = os.path.join(OUT_TXT, 'checkpoint_artistlist1_image0.json')

recs_txt = json.load(open(CKPT_TXT)) if os.path.exists(CKPT_TXT) else []
done = {(r['sample_id'], r['phrasing']) for r in recs_txt}
todo = [(sid, ph) for sid in SAMPLE_IDS for ph in make_phrasings(DESIGN_BY_ID[sid]) if (sid, ph) not in done]
print(f'{len(done)} done, {len(todo)} to run')

def _save():
    tmp = CKPT_TXT + '.tmp'; json.dump(recs_txt, open(tmp, 'w')); os.replace(tmp, CKPT_TXT)

t0 = time.time()
for n, (sid, ph) in enumerate(todo, 1):
    row = DESIGN_BY_ID[sid]
    request = make_phrasings(row)[ph]
    rec = {'sample_id': sid, 'category': PREFIX_TO_CATEGORY[sid[0].upper()], 'phrasing': ph, 'request': request,
           'artist_list_given': True, 'use_image': False,
           'gold_region1': row['region1'], 'gold_style1': row['style1'],
           'gold_region2': row['region2'], 'gold_style2': row['style2']}
    try:
        raw, lat = call_qwen_text(request)
        rec.update({'raw_output': raw, 'latency_s': lat})
    except Exception as e:
        rec.update({'raw_output': None, 'latency_s': None, 'error': f'{type(e).__name__}: {e}'})
    recs_txt.append(rec)
    if n % 20 == 0 or n == len(todo):
        _save(); print(f'{n}/{len(todo)} | {time.time() - t0:.0f}s | checkpoint saved')
_save()

df_txt = pd.DataFrame(recs_txt)
df_txt.to_csv(os.path.join(OUT_TXT, 'results_artistlist1_image0.csv'), index=False)
print('saved', len(df_txt), 'rows to', OUT_TXT)

0 done, 800 to run
20/800 | 54s | checkpoint saved
40/800 | 110s | checkpoint saved
60/800 | 165s | checkpoint saved
80/800 | 220s | checkpoint saved
100/800 | 276s | checkpoint saved
120/800 | 331s | checkpoint saved
140/800 | 387s | checkpoint saved
160/800 | 442s | checkpoint saved
180/800 | 498s | checkpoint saved
200/800 | 552s | checkpoint saved
220/800 | 607s | checkpoint saved
240/800 | 662s | checkpoint saved
260/800 | 717s | checkpoint saved
280/800 | 772s | checkpoint saved
300/800 | 826s | checkpoint saved
320/800 | 883s | checkpoint saved
340/800 | 937s | checkpoint saved
360/800 | 991s | checkpoint saved
380/800 | 1046s | checkpoint saved
400/800 | 1102s | checkpoint saved
420/800 | 1156s | checkpoint saved
440/800 | 1211s | checkpoint saved
460/800 | 1266s | checkpoint saved
480/800 | 1322s | checkpoint saved
500/800 | 1377s | checkpoint saved
520/800 | 1434s | checkpoint saved
540/800 | 1489s | checkpoint saved
560/800 | 1544s | checkpoint saved
580/800 | 1599s | checkp

In [ ]:

import os, numpy as np, pandas as pd

_missing = [n for n in ('exact_under', 'region_word', 'cluster_ci', 'region_strict', 'region_lenient') if n not in globals()]
assert not _missing, f'Run Patch 1 first (missing: {_missing})'

IMG_CSV = '/content/drive/MyDrive/qwen_free_form_test_200/results_artistlist1.csv'
df_img = pd.read_csv(IMG_CSV) if os.path.exists(IMG_CSV) else df.copy()      # image run
df_txt = df_txt.copy() if 'df_txt' in globals() else pd.read_csv('/content/drive/MyDrive/qwen_free_form_test_200_textonly/results_artistlist1_image0.csv')

for d in (df_img, df_txt):
    d['ex_strict']  = d.apply(lambda r: exact_under(r, region_strict), axis=1)
    d['ex_word']    = d.apply(lambda r: exact_under(r, region_word), axis=1)
    d['ex_lenient'] = d.apply(lambda r: exact_under(r, region_lenient), axis=1)

def swapped(r):
    obj, _ = extract_json(r['raw_output']) if isinstance(r['raw_output'], str) else (None, None)
    asg = get_assignments(obj) if obj is not None else None
    if not asg or len(asg) != 2 or r['gold_style1'] == r['gold_style2']:
        return False
    i1 = next((k for k, p in enumerate(asg) if region_lenient(p['region'], r['gold_region1'])), None)
    i2 = next((k for k, p in enumerate(asg) if region_lenient(p['region'], r['gold_region2']) and k != i1), None)
    if i1 is None or i2 is None:
        return False
    return canon_artist(asg[i1]['artist']) == r['gold_style2'] and canon_artist(asg[i2]['artist']) == r['gold_style1']

def unmatched_regions(r):     # predicted regions matching neither gold region
    obj, _ = extract_json(r['raw_output']) if isinstance(r['raw_output'], str) else (None, None)
    asg = get_assignments(obj) if obj is not None else None
    return sum(not any(region_lenient(p['region'], g) for g in (r['gold_region1'], r['gold_region2'])) for p in (asg or []))

for d in (df_img, df_txt):
    d['swapped'] = d.apply(swapped, axis=1)
    d['unmatched'] = d.apply(unmatched_regions, axis=1)

m = df_img.merge(df_txt, on=['sample_id', 'phrasing'], suffixes=('_img', '_txt'))
print(f'{len(m)} matched calls ({m.sample_id.nunique()} samples)\n')

rng2 = np.random.default_rng(42)
def paired_ci(col, B=5000):
    m['d'] = m[col + '_txt'].astype(int) - m[col + '_img'].astype(int)
    per = m.groupby('sample_id')['d'].agg(['sum', 'count'])
    s, c = per['sum'].to_numpy(float), per['count'].to_numpy(float)
    idx = rng2.integers(0, len(s), size=(B, len(s)))
    est = s[idx].sum(1) / c[idx].sum(1)
    return 100 * s.sum() / c.sum(), 100 * np.percentile(est, 2.5), 100 * np.percentile(est, 97.5)

print('EXACT MATCH')
for col, label in (('ex_strict', 'strict'), ('ex_word', 'word-level'), ('ex_lenient', 'lenient (main rule)')):
    pi = 100 * m[col + '_img'].mean(); pt = 100 * m[col + '_txt'].mean()
    d, lo, hi = paired_ci(col)
    print(f'  {label:20s} image {pi:5.1f}%   text-only {pt:5.1f}%   text-only minus image {d:+.1f} pts  [{lo:+.1f}, {hi:+.1f}] (sample-level)')

print('\nBY PHRASING (main rule)')
print(pd.DataFrame({'image': df_img.groupby('phrasing').ex_lenient.mean(),
                    'text-only': df_txt.groupby('phrasing').ex_lenient.mean()}).round(3).to_string())

print('\nSWAPPED STYLES  image:', int(df_img.swapped.sum()), '| text-only:', int(df_txt.swapped.sum()), '(of', len(df_img), ')')
print('PREDICTED REGIONS MATCHING NEITHER GOLD REGION  image:', int(df_img.unmatched.sum()), '| text-only:', int(df_txt.unmatched.sum()))

print('\ncalls by (image correct, text-only correct):')
print(pd.crosstab(m.ex_lenient_img, m.ex_lenient_txt, rownames=['image'], colnames=['text-only']).to_string())

hurt = m[(~m.ex_lenient_img) & (m.ex_lenient_txt)]
print(f'\nimage wrong but text-only right: {len(hurt)} calls;  image right but text-only wrong: {int(((m.ex_lenient_img) & (~m.ex_lenient_txt)).sum())} calls')
for _, r in hurt.head(8).iterrows():
    print(f"  [{r.sample_id}/{r.phrasing}] image raw: {str(r.raw_output_img)[:170]}")

800 matched calls (200 samples)

EXACT MATCH
  strict               image  84.9%   text-only  86.5%   text-only minus image +1.6 pts  [-1.6, +5.0] (sample-level)
  word-level           image  86.2%   text-only  86.5%   text-only minus image +0.2 pts  [-2.9, +3.5] (sample-level)
  lenient (main rule)  image  86.5%   text-only  86.5%   text-only minus image +0.0 pts  [-3.1, +3.2] (sample-level)

BY PHRASING (main rule)
                 image  text-only
phrasing                         
casual_fullname   0.79       1.00
direct            0.85       1.00
reversed          0.94       1.00
style_first       0.88       0.46

SWAPPED STYLES  image: 32 | text-only: 4 (of 800 )
PREDICTED REGIONS MATCHING NEITHER GOLD REGION  image: 84 | text-only: 117

calls by (image correct, text-only correct):
text-only  False  True 
image                  
False         20     88
True          88    604

image wrong but text-only right: 88 calls;  image right but text-only wrong: 88 calls
  [C003/direct] ima